In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
generator_llm = ChatOllama(model="llama3.2")
evaluator_llm = ChatOllama(model="llama3.2")
optimizer_llm = ChatOllama(model="llama3.2")


In [3]:
#state
class TweetState(TypedDict): 
    topic: str 
    tweet: str
    evaluation: Literal["approved", "needs_improvements"]
    feedback: str 
    iteration: int 
    max_iteration: int

In [ ]:
def generate_tweet(state: TweetState):
    #prompt 
    messages = [
        SystemMessage(content="You are a funny and clever Twetter/X influencer."),
        HumanMessage(content=f"""Write a short original, and hailarious tweet on the topiv: {state['topic']}
        Rules:
        -Do  not use qestion answer format
        - Mx 280 cgaracters
        -use observationa; humor, irnoy, sarcasm, or cultural references.
        -Thinj in meme logic, puchlines ir relatables takes.
        -use simpel day to day englifh
        - this is a version {state['iteration']+1}""")
    ]
    #send generator_llm
    response = generator_llm.invoke(messages).content

    #return response
    return{"tweet": response}

In [11]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel): 
    evaluation: Literal['approved', 'needs_improvement'] = Field(..., description="Final evaluarion result.")
    feedback: str = Field(..., description="Constructive feedback for the tweet.")
    score: int = Field(..., ge=0, le=5, description="Total score from rubric (0 to 5)")

In [8]:
def evaluate_tweet(state: TweetState):

    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [9]:
graph = StateGraph(TweetState)

graph.add_node("generate",generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)


NameError: name 'optimize_tweet' is not defined